# Walkthrough: Two-Tower Dual-Encoder Document Retrieval

This notebook narrates the same five pipeline stages the README's Usage section documents,
calling the real package modules directly rather than reimplementing any of their logic.

```
cbow (tokenizer + embeddings) -> MS MARCO triples -> tokenize -> GRU tower training
  -> index corpus in Redis -> interactive ANN query
```

See README sections 2-3 for the method and architecture diagrams behind each stage, and
section 6 for full-scale results. A **local Redis with the RediSearch module** must already be
running before Steps 4-5 (`make redis`, or `docker run -d --name ai-residency-week2-redis -p 6390:6379
redis/redis-stack-server:latest redis-stack-server --save "" --appendonly no`), with `.env` /
environment variables `REDIS_HOST` / `REDIS_PORT` / `REDIS_PASSWORD` pointing at it.

In [ ]:
import os

# Every path in the package resolves relative to the repo root (data/, checkpoints/), matching
# the README and Makefile, so move the kernel's cwd up one level. Idempotent: cwd stops being
# "notebooks" after the first run. There is no import-path juggling any more -- the flat
# modules at the repo root import directly, so `import config` and friends just work.
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

os.environ.setdefault("WANDB_MODE", "offline")

from config import Config

config = Config()
print(config.data_dir, config.checkpoint_dir)

## Step 1 -- CBOW word embeddings

Trains the tokenizer/vocab and the CBOW embeddings that both towers sit on top of, frozen.
A full-scale run (100MB `text8`) takes hours end to end; this is fine to run at reduced scale
for a quick demo and still exercises the exact same code path.

In [ ]:
from cbow.train_tokenizer import (
    main as build_vocabulary,
)
from cbow.train_word2vec import train_word2vec

build_vocabulary()
train_word2vec(config)

## Step 2 -- Build + tokenize MS MARCO triples

`generate_triples()` pulls real MS MARCO v1.1 queries/passages/relevance labels and builds
`(query, positive_doc, negative_doc)` triples; `process_triples()` tokenizes them with the CBOW
vocabulary just trained above. Both are called here as real importable functions, not shelled out.
The cell then truncates the tokenized output to a small demo-scale slice -- see the comment in
the cell below for why.

In [ ]:
from data_prep.generate_triples import (
    generate_triples,
)
from data_prep.tokenize_triples import (
    tokenize_triples,
)

generate_triples(config)
tokenize_triples(config)

# generate_triples() just built the real, full MS MARCO v1.1 triple set (88,523 / 10,783 /
# 10,448 train/validation/test) -- not fabricated, not reduced. But Step 3's unbatched
# per-example GRU training loop benchmarks at ~23ms/triple, so training on the full train split
# takes ~5.6 hours end to end -- too long for a notebook walkthrough meant to run in minutes.
# Truncating to 500/500/500 here matches the same reduced scale the README's own Experimental
# Results section trains and reports numbers on, and still exercises the identical training code
# path Step 3 below calls. Both files are gitignored and regenerable in under a minute (this
# cell), so this truncation is a data-only change -- delete the block below and re-run for a
# genuine full-scale training run.
import json

with open(config.tokenized_triples_path) as handle:
    tokenized = json.load(handle)
for split in ["train", "validation", "test"]:
    tokenized[split] = tokenized[split][:500]
with open(config.tokenized_triples_path, "w") as handle:
    json.dump(tokenized, handle)
print({k: len(v) for k, v in tokenized.items()})

## Step 3 -- Train GRU dual-encoder towers

`train()` runs the triplet-loss training loop described in README section 2.3, scores a
never-trained pair of towers on the held-out validation triples first as a baseline, and
persists `checkpoints/qry_tower.pth` / `checkpoints/doc_tower.pth`. Tracking falls back to
offline mode without a `WANDB_API_KEY`, set in the first cell.

In [ ]:
from training import train

result = train(config)
result

## Step 4 -- Index the corpus in Redis

Requires the Redis + RediSearch container mentioned above to already be running. Every unique
positive document is encoded once with the trained document tower and written into the vector
index, which is the whole reason a dual encoder is worth training: query time then costs one
encode plus an ANN lookup.

In [ ]:
from retrieval.query_engine import get_redis_client
from retrieval.save_doc_embeddings_to_redis import (
    index_documents,
)

index_documents(config, get_redis_client(config))

## Step 5 -- Query

`run_query()` is what the CLI, the FastAPI service and the Streamlit UI all call, so this is the
same retrieval path all three take, not a notebook-only reimplementation.

In [ ]:
from retrieval.query_engine import run_query

for rank, (score, text, _embedding) in enumerate(
    run_query("was ronald reagan a democrat"), start=1
):
    print(f"Rank {rank}  distance={score:.4f}\n{text[:200]}...\n---")

## Wrap-up

That's the full pipeline: frozen word embeddings, learned GRU towers, triplet-loss training,
and Redis/RediSearch as the ANN index at serving time. See the README's Experimental Results
section for real training-loss numbers, a sample query transcript, and an honest account of
what the held-out pairwise accuracy says about how much the towers actually learned.